### Retrieve secrets and verify SPN via Spark Conf

In [0]:
SCOPE_NAME = "daniloshurko-kv-scope"
STORAGE_ACCOUNT = "dlsua5816bd"
CONTAINER = "daniloshurko"

client_id = dbutils.secrets.get(scope=SCOPE_NAME, key="sp-databricks-adls-appid")
client_secret = dbutils.secrets.get(scope=SCOPE_NAME, key="sp-databricks-adls-appkey")
tenant_id = dbutils.secrets.get(scope=SCOPE_NAME, key="tenant-id")

spark.conf.set(f"fs.azure.account.auth.type.{STORAGE_ACCOUNT}.dfs.core.windows.net", "OAuth")
spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
)
spark.conf.set(f"fs.azure.account.oauth2.client.id.{STORAGE_ACCOUNT}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{STORAGE_ACCOUNT}.dfs.core.windows.net", client_secret)
spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    f"https://login.microsoftonline.com/{tenant_id}/oauth2/token",
)

base_path = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"
display(dbutils.fs.ls(base_path))

### Demonstrate legacy mount approach

In [0]:
configs = {
    "fs.azure.account.auth.type": "OAuth",
    "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    "fs.azure.account.oauth2.client.id": client_id,
    "fs.azure.account.oauth2.client.secret": client_secret,
    "fs.azure.account.oauth2.client.endpoint": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token",
}

mount_point = f"/mnt/{CONTAINER}"

try:
    dbutils.fs.mount(
        source=base_path,
        mount_point=mount_point,
        extra_configs=configs
    )
    print(f"Mounted successfully to {mount_point}")
except Exception as e:
    print(f"Mount failed as expected on Unity Catalog Shared cluster: {e}")